In [1]:
# Cell A
import numpy as np
from scipy.sparse.linalg import LinearOperator, eigsh
np.set_printoptions(precision=6, suppress=True)

In [2]:
# Cell B
def spin_half_ops(dtype=np.complex128):
    I = np.eye(2, dtype=dtype)
    Sp = np.array([[0, 1],
                   [0, 0]], dtype=dtype)
    Sm = np.array([[0, 0],
                   [1, 0]], dtype=dtype)
    Sz = 0.5 * np.array([[1, 0],
                         [0, -1]], dtype=dtype)
    return I, Sp, Sm, Sz

def spin_one_ops(dtype=np.complex128):
    I = np.eye(3, dtype=dtype)
    Sp = np.sqrt(2) * np.array([[0, 1, 0],
                                [0, 0, 1],
                                [0, 0, 0]], dtype=dtype)
    Sm = np.sqrt(2) * np.array([[0, 0, 0],
                                [1, 0, 0],
                                [0, 1, 0]], dtype=dtype)
    Sz = np.array([[1, 0, 0],
                   [0, 0, 0],
                   [0, 0, -1]], dtype=dtype)
    return I, Sp, Sm, Sz

In [3]:
# Cell C
def heisenberg_mpo(L, J=1.0, Jz=1.0, h=0.0, dtype=np.complex128):
    I, Sp, Sm, Sz = spin_half_ops(dtype=dtype)
    d = 2
    chi = 5

    Wbulk = np.zeros((chi, chi, d, d), dtype=dtype)
    Wbulk[0, 0] = I
    Wbulk[1, 0] = Sp
    Wbulk[2, 0] = Sm
    Wbulk[3, 0] = Sz
    Wbulk[4, 0] = -h * Sz
    Wbulk[4, 1] = (J / 2.0) * Sm
    Wbulk[4, 2] = (J / 2.0) * Sp
    Wbulk[4, 3] = Jz * Sz
    Wbulk[4, 4] = I

    Wleft = np.zeros((1, chi, d, d), dtype=dtype)
    Wleft[0, 0] = -h * Sz
    Wleft[0, 1] = (J / 2.0) * Sm
    Wleft[0, 2] = (J / 2.0) * Sp
    Wleft[0, 3] = Jz * Sz
    Wleft[0, 4] = I

    Wright = np.zeros((chi, 1, d, d), dtype=dtype)
    Wright[0, 0] = I
    Wright[1, 0] = Sp
    Wright[2, 0] = Sm
    Wright[3, 0] = Sz
    Wright[4, 0] = -h * Sz

    mpo = [None] * L
    mpo[0] = Wleft
    for i in range(1, L - 1):
        mpo[i] = Wbulk.copy()
    mpo[L - 1] = Wright
    return mpo

def aklt_mpo(L, dtype=np.complex128):
    I, Sp, Sm, Sz = spin_one_ops(dtype=dtype)
    d = 3
    chi = 14

    O = [Sp / np.sqrt(2.0), Sm / np.sqrt(2.0), Sz]
    Obar = [Sm / np.sqrt(2.0), Sp / np.sqrt(2.0), Sz]

    O2 = []
    Obar2 = []
    for a in range(3):
        for b in range(3):
            O2.append(O[a] @ O[b])
            Obar2.append(Obar[a] @ Obar[b])

    Wbulk = np.zeros((chi, chi, d, d), dtype=dtype)
    Wbulk[0, 0] = I
    for a in range(3):
        Wbulk[0, 1 + a] = O[a]
    for ab in range(9):
        Wbulk[0, 4 + ab] = (1.0 / 3.0) * O2[ab]
    for a in range(3):
        Wbulk[1 + a, 13] = Obar[a]
    for ab in range(9):
        Wbulk[4 + ab, 13] = Obar2[ab]
    Wbulk[13, 13] = I

    Wleft = np.zeros((1, chi, d, d), dtype=dtype)
    Wleft[0, 0] = I
    for a in range(3):
        Wleft[0, 1 + a] = O[a]
    for ab in range(9):
        Wleft[0, 4 + ab] = (1.0 / 3.0) * O2[ab]

    Wright = np.zeros((chi, 1, d, d), dtype=dtype)
    for a in range(3):
        Wright[1 + a, 0] = Obar[a]
    for ab in range(9):
        Wright[4 + ab, 0] = Obar2[ab]
    Wright[13, 0] = I

    mpo = [None] * L
    mpo[0] = Wleft
    for i in range(1, L - 1):
        mpo[i] = Wbulk.copy()
    mpo[L - 1] = Wright
    return mpo

In [4]:
# Cell D
def kron_all(ops):
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out

def onsite_op(L, i, op, d):
    I = np.eye(d, dtype=np.complex128)
    ops = [I] * L
    ops[i] = op
    return kron_all(ops)

def bond_op(L, i, op_i, op_j, d):
    I = np.eye(d, dtype=np.complex128)
    ops = [I] * L
    ops[i] = op_i
    ops[i + 1] = op_j
    return kron_all(ops)

def dense_heisenberg(L, J=1.0, Jz=1.0, h=0.0):
    I, Sp, Sm, Sz = spin_half_ops()
    d = 2
    H = np.zeros((d**L, d**L), dtype=np.complex128)

    for i in range(L - 1):
        H += (J / 2.0) * bond_op(L, i, Sp, Sm, d)
        H += (J / 2.0) * bond_op(L, i, Sm, Sp, d)
        H += Jz * bond_op(L, i, Sz, Sz, d)

    for i in range(L):
        H += -h * onsite_op(L, i, Sz, d)

    return H

def dense_aklt(L):
    I, Sp, Sm, Sz = spin_one_ops()
    d = 3
    H = np.zeros((d**L, d**L), dtype=np.complex128)

    for i in range(L - 1):
        X = (
            0.5 * bond_op(L, i, Sp, Sm, d)
            + 0.5 * bond_op(L, i, Sm, Sp, d)
            + bond_op(L, i, Sz, Sz, d)
        )
        H += X + (1.0 / 3.0) * (X @ X)

    return H

In [5]:
# Cell E
def random_mps(L, d, Dmax, seed=1234, dtype=np.complex128):
    rng = np.random.default_rng(seed)

    dims = [1]
    for i in range(1, L):
        dims.append(min(Dmax, d ** min(i, L - i)))
    dims.append(1)

    mps = []
    for i in range(L):
        Dl, Dr = dims[i], dims[i + 1]
        A = rng.normal(size=(Dl, d, Dr)) + 1j * rng.normal(size=(Dl, d, Dr))
        mps.append(A.astype(dtype))
    return mps


def mps_to_state(mps):
    psi = mps[0][0]
    for i in range(1, len(mps)):
        psi = np.einsum('...a,asb->...sb', psi, mps[i], optimize=True)
    return psi.reshape(-1, order='C')


def normalize_mps_by_state(mps):
    psi = mps_to_state(mps)
    nrm = np.sqrt(np.vdot(psi, psi).real)
    mps = [A.copy() for A in mps]
    mps[0] = mps[0] / nrm
    return mps


def left_basis_map(mps, i):
    if i == 0:
        return np.array([[1.0]], dtype=np.complex128)

    X = mps[0][0]
    for k in range(1, i):
        X = np.einsum('pa,asb->psb', X, mps[k], optimize=True)
        X = X.reshape((-1, X.shape[-1]), order='C')
    return X


def right_basis_map(mps, i):
    L = len(mps)
    if i == L - 2:
        return np.array([[1.0]], dtype=np.complex128)

    X = mps[i + 2]
    for k in range(i + 3, L):
        X = np.einsum('asd,dtu->astu', X, mps[k], optimize=True)
        X = X.reshape((X.shape[0], -1, X.shape[-1]), order='C')
    return X[:, :, 0]


def make_mixed_canonical_around_bond(mps, i):
    mps = [A.copy() for A in mps]
    L = len(mps)

    for k in range(i):
        A = mps[k]
        Dl, d, Dr = A.shape
        M = A.reshape((Dl * d, Dr), order='C')
        Q, R = np.linalg.qr(M, mode='reduced')
        nk = Q.shape[1]
        mps[k] = Q.reshape((Dl, d, nk), order='C')
        mps[k + 1] = np.einsum('ab,bsd->asd', R, mps[k + 1], optimize=True)

    for k in range(L - 1, i + 1, -1):
        A = mps[k]
        Dl, d, Dr = A.shape
        M = A.reshape((Dl, d * Dr), order='C')
        Q, R = np.linalg.qr(M.conj().T, mode='reduced')
        B = Q.conj().T.reshape((-1, d, Dr), order='C')
        mps[k] = B
        Rd = R.conj().T
        mps[k - 1] = np.einsum('xsa,ab->xsb', mps[k - 1], Rd, optimize=True)

    return mps

In [6]:
# Cell F
def init_left_env():
    Lenv = np.zeros((1, 1, 1), dtype=np.complex128)
    Lenv[0, 0, 0] = 1.0
    return Lenv


def init_right_env():
    Renv = np.zeros((1, 1, 1), dtype=np.complex128)
    Renv[0, 0, 0] = 1.0
    return Renv


def left_env_update(Lold, A, W):
    return np.einsum('bxy,xsa,bBst,ytc->Bac', Lold, A, W, A.conj(), optimize=True)


def right_env_update(Rold, B, W):
    return np.einsum('xsa,bBst,Bac,ytc->bxy', B, W, Rold, B.conj(), optimize=True)


def build_left_envs(mps, mpo):
    L = len(mps)
    Lenvs = [None] * L
    Lenvs[0] = init_left_env()
    cur = Lenvs[0]
    for i in range(0, L - 1):
        cur = left_env_update(cur, mps[i], mpo[i])
        Lenvs[i + 1] = cur
    return Lenvs


def build_right_envs(mps, mpo):
    L = len(mps)
    Renvs = [None] * L
    Renvs[L - 1] = init_right_env()
    cur = Renvs[L - 1]
    for i in range(L - 1, 0, -1):
        cur = right_env_update(cur, mps[i], mpo[i])
        Renvs[i - 1] = cur
    return Renvs

In [7]:
# Cell G
def apply_one_site_heff_env(M, Lenv, W, Renv):
    X = np.einsum('byx,ysz->bxsz', Lenv, M, optimize=True)
    Y = np.einsum('bBst,bxsz->Bxtz', W, X, optimize=True)
    HM = np.einsum('Bxtz,Bza->xta', Y, Renv, optimize=True)
    return HM


def apply_two_site_heff_env(theta, Lenv, W1, W2, Renv):
    X = np.einsum('byx,yuvz->bxuvz', Lenv, theta, optimize=True)
    Y = np.einsum('bBus,bxuvz->Bxsvz', W1, X, optimize=True)
    Z = np.einsum('BCvt,Bxsvz->Cxstz', W2, Y, optimize=True)
    HT = np.einsum('Cxstz,Cza->xsta', Z, Renv, optimize=True)
    return HT


def heff_two_site_env_matvec(v, Lenv, W1, W2, Renv, Dl, d1, d2, Dr):
    theta = v.reshape((Dl, d1, d2, Dr), order='C')
    out = apply_two_site_heff_env(theta, Lenv, W1, W2, Renv)
    return out.reshape(Dl * d1 * d2 * Dr, order='C')

In [8]:
# Cell H
def dense_projected_two_site_heff_from_full(Hfull, mps, i, d):
    UL = left_basis_map(mps, i)
    UR = right_basis_map(mps, i)

    Dl = UL.shape[1]
    Dr = UR.shape[0]
    dimL = UL.shape[0]
    dimR = UR.shape[1]

    Nloc = Dl * d * d * Dr
    Dim = dimL * d * d * dimR
    P = np.zeros((Dim, Nloc), dtype=np.complex128)

    col = 0
    for a in range(Dl):
        for s in range(d):
            for t in range(d):
                for c in range(Dr):
                    mid = np.zeros(d * d, dtype=np.complex128)
                    mid[s * d + t] = 1.0
                    vec = np.kron(UL[:, a], np.kron(mid, UR[c, :]))
                    P[:, col] = vec
                    col += 1

    Heff = P.conj().T @ Hfull @ P
    return Heff

def build_dense_local_heff_from_env(mps, mpo, i):
    A = mps[i]
    B = mps[i + 1]
    Dl, d1, mid = A.shape
    mid2, d2, Dr = B.shape
    assert mid == mid2

    Lenvs = build_left_envs(mps, mpo)
    Renvs = build_right_envs(mps, mpo)

    Lenv = Lenvs[i]
    Renv = Renvs[i + 1]

    N = Dl * d1 * d2 * Dr
    Heff = np.zeros((N, N), dtype=np.complex128)

    for j in range(N):
        e = np.zeros(N, dtype=np.complex128)
        e[j] = 1.0
        theta = e.reshape((Dl, d1, d2, Dr), order='C')
        Heff[:, j] = apply_two_site_heff_env(
            theta, Lenv, mpo[i], mpo[i + 1], Renv
        ).reshape(-1, order='C')

    return Heff

def compare_two_site_env_vs_proj(Hfull, mpo, mps, i, d):
    Heff_proj = dense_projected_two_site_heff_from_full(Hfull, mps, i, d)
    Heff_env = build_dense_local_heff_from_env(mps, mpo, i)

    diff = np.max(np.abs(Heff_env - Heff_proj))
    herm_env = np.max(np.abs(Heff_env - Heff_env.conj().T))
    herm_proj = np.max(np.abs(Heff_proj - Heff_proj.conj().T))

    print(f"bond {i}: max|env-proj| = {diff:.3e}")
    print(f"bond {i}: env Herm err = {herm_env:.3e}")
    print(f"bond {i}: proj Herm err = {herm_proj:.3e}")

In [9]:
# Cell I
def form_theta(A, B):
    return np.einsum('asm,mtr->astr', A, B, optimize=True)

def split_theta_left_to_right(theta, Dmax, svd_cutoff=1e-12):
    Dl, d1, d2, Dr = theta.shape
    M = theta.reshape((Dl * d1, d2 * Dr), order='C')

    U, S, Vh = np.linalg.svd(M, full_matrices=False)

    keep = np.sum(S > svd_cutoff)
    keep = max(1, min(Dmax, keep, len(S)))
    disc_weight = np.sum(S[keep:]**2).real

    U = U[:, :keep]
    S = S[:keep]
    Vh = Vh[:keep, :]

    A = U.reshape((Dl, d1, keep), order='C')
    B = (np.diag(S) @ Vh).reshape((keep, d2, Dr), order='C')
    return A, B, S, disc_weight

def split_theta_right_to_left(theta, Dmax, svd_cutoff=1e-12):
    Dl, d1, d2, Dr = theta.shape
    M = theta.reshape((Dl * d1, d2 * Dr), order='C')

    U, S, Vh = np.linalg.svd(M, full_matrices=False)

    keep = np.sum(S > svd_cutoff)
    keep = max(1, min(Dmax, keep, len(S)))
    disc_weight = np.sum(S[keep:]**2).real

    U = U[:, :keep]
    S = S[:keep]
    Vh = Vh[:keep, :]

    A = (U @ np.diag(S)).reshape((Dl, d1, keep), order='C')
    B = Vh.reshape((keep, d2, Dr), order='C')
    return A, B, S, disc_weight

def expectation_value_mpo(mps, mpo):
    env = init_left_env()
    for i in range(len(mps)):
        env = left_env_update(env, mps[i], mpo[i])
    return env[0, 0, 0]

In [10]:
# Cell J
def exact_aklt_mps(L, left_vec=None, right_vec=None, dtype=np.complex128):
    Ap = np.array([[0, np.sqrt(2/3)],
                   [0, 0]], dtype=dtype)
    A0 = np.array([[-1/np.sqrt(3), 0],
                   [0, 1/np.sqrt(3)]], dtype=dtype)
    Am = np.array([[0, 0],
                   [-np.sqrt(2/3), 0]], dtype=dtype)

    local = np.stack([Ap, A0, Am], axis=1)

    if left_vec is None:
        left_vec = np.array([1.0, 0.0], dtype=dtype)
    if right_vec is None:
        right_vec = np.array([1.0, 0.0], dtype=dtype)

    mps = []
    A1 = np.einsum('a,asb->sb', left_vec, local).reshape(1, 3, 2)
    mps.append(A1)

    for _ in range(1, L - 1):
        mps.append(local.copy())

    AL = np.einsum('asb,b->as', local, right_vec).reshape(2, 3, 1)
    mps.append(AL)

    return mps

In [11]:
# Cell K
def two_site_dmrg_env_matrix_free(
    mpo,
    d,
    Dmax=32,
    nsweeps=8,
    lanczos_tol=1e-10,
    lanczos_maxiter=None,
    svd_cutoff=1e-12,
    seed=1234,
    verbose=True,
):
    L = len(mpo)

    mps = random_mps(L, d=d, Dmax=min(Dmax, 4), seed=seed)
    mps = normalize_mps_by_state(mps)
    mps = make_mixed_canonical_around_bond(mps, 0)

    sweep_local_energies = []
    sweep_discards = []

    for sweep in range(nsweeps):
        discards_this_sweep = []
        last_local_energy = None

        # Left -> Right
        Renvs = build_right_envs(mps, mpo)
        Lenv = init_left_env()

        for i in range(L - 1):
            theta0 = form_theta(mps[i], mps[i + 1])
            Dl, d1, d2, Dr = theta0.shape
            Nloc = Dl * d1 * d2 * Dr
            Renv = Renvs[i + 1]

            Heff = LinearOperator(
                shape=(Nloc, Nloc),
                matvec=lambda v, Lenv=Lenv, W1=mpo[i], W2=mpo[i + 1], Renv=Renv,
                              Dl=Dl, d1=d1, d2=d2, Dr=Dr:
                    heff_two_site_env_matvec(v, Lenv, W1, W2, Renv, Dl, d1, d2, Dr),
                dtype=np.complex128
            )

            v0 = theta0.reshape(-1, order='C')
            vals, vecs = eigsh(
                Heff, k=1, which='SA',
                v0=v0, tol=lanczos_tol, maxiter=lanczos_maxiter
            )
            last_local_energy = vals[0].real

            theta = vecs[:, 0].reshape((Dl, d1, d2, Dr), order='C')
            Anew, Bnew, S, disc = split_theta_left_to_right(theta, Dmax, svd_cutoff)

            mps[i] = Anew
            mps[i + 1] = Bnew
            discards_this_sweep.append(disc)

            Lenv = left_env_update(Lenv, mps[i], mpo[i])

        mps = normalize_mps_by_state(mps)
        mps = make_mixed_canonical_around_bond(mps, L - 2)

        # Right -> Left
        Lenvs = build_left_envs(mps, mpo)
        Renv = init_right_env()

        for i in range(L - 2, -1, -1):
            theta0 = form_theta(mps[i], mps[i + 1])
            Dl, d1, d2, Dr = theta0.shape
            Nloc = Dl * d1 * d2 * Dr
            Lcur = Lenvs[i]

            Heff = LinearOperator(
                shape=(Nloc, Nloc),
                matvec=lambda v, Lcur=Lcur, W1=mpo[i], W2=mpo[i + 1], Renv=Renv,
                              Dl=Dl, d1=d1, d2=d2, Dr=Dr:
                    heff_two_site_env_matvec(v, Lcur, W1, W2, Renv, Dl, d1, d2, Dr),
                dtype=np.complex128
            )

            v0 = theta0.reshape(-1, order='C')
            vals, vecs = eigsh(
                Heff, k=1, which='SA',
                v0=v0, tol=lanczos_tol, maxiter=lanczos_maxiter
            )
            last_local_energy = vals[0].real

            theta = vecs[:, 0].reshape((Dl, d1, d2, Dr), order='C')
            Anew, Bnew, S, disc = split_theta_right_to_left(theta, Dmax, svd_cutoff)

            mps[i] = Anew
            mps[i + 1] = Bnew
            discards_this_sweep.append(disc)

            Renv = right_env_update(Renv, mps[i + 1], mpo[i + 1])

        mps = normalize_mps_by_state(mps)
        mps = make_mixed_canonical_around_bond(mps, 0)

        max_disc = max(discards_this_sweep) if discards_this_sweep else 0.0
        sweep_local_energies.append(last_local_energy if last_local_energy is not None else np.nan)
        sweep_discards.append(max_disc)

        if verbose:
            print(f"Sweep {sweep+1:2d}: local eig E = {sweep_local_energies[-1]:.12f}, "
                  f"max discarded weight = {max_disc:.3e}")

    return mps, np.array(sweep_local_energies), np.array(sweep_discards)

In [12]:
# Cell L
L = 4
d = 2
i = 1

mpo = heisenberg_mpo(L)
Hfull = dense_heisenberg(L)

mps = random_mps(L, d=d, Dmax=3, seed=7)
mps = normalize_mps_by_state(mps)
mps = make_mixed_canonical_around_bond(mps, i)

compare_two_site_env_vs_proj(Hfull, mpo, mps, i, d)

bond 1: max|env-proj| = 1.665e-16
bond 1: env Herm err = 0.000e+00
bond 1: proj Herm err = 5.551e-17


In [13]:
# Cell M
L = 8
mpo_h = heisenberg_mpo(L, J=1.0, Jz=1.0, h=0.0)

mps_h, Es_h, disc_h = two_site_dmrg_env_matrix_free(
    mpo=mpo_h,
    d=2,
    Dmax=32,
    nsweeps=6,
    seed=1234,
    verbose=True
)

E_mpo_h = expectation_value_mpo(mps_h, mpo_h).real
print("\nHeisenberg <H> from MPO contraction =", E_mpo_h)

Hfull_h = dense_heisenberg(L, J=1.0, Jz=1.0, h=0.0)
psi_h = mps_to_state(mps_h)
E_dense_h = (np.vdot(psi_h, Hfull_h @ psi_h) / np.vdot(psi_h, psi_h)).real
E0_h = np.linalg.eigvalsh(Hfull_h)[0].real

print("Heisenberg energy from dense H on final MPS =", E_dense_h)
print("Heisenberg exact ED ground energy           =", E0_h)
print("Absolute error                              =", abs(E_dense_h - E0_h))

Sweep  1: local eig E = -3.374932598688, max discarded weight = 0.000e+00
Sweep  2: local eig E = -3.374932598688, max discarded weight = 0.000e+00
Sweep  3: local eig E = -3.374932598688, max discarded weight = 0.000e+00
Sweep  4: local eig E = -3.374932598688, max discarded weight = 0.000e+00
Sweep  5: local eig E = -3.374932598688, max discarded weight = 0.000e+00
Sweep  6: local eig E = -3.374932598688, max discarded weight = 0.000e+00

Heisenberg <H> from MPO contraction = -3.374932598687892
Heisenberg energy from dense H on final MPS = -3.37493259868789
Heisenberg exact ED ground energy           = -3.374932598687897
Absolute error                              = 6.661338147750939e-15


In [14]:
# Cell N
L = 6
mpo_a = aklt_mpo(L)

mps_a, Es_a, disc_a = two_site_dmrg_env_matrix_free(
    mpo=mpo_a,
    d=3,
    Dmax=16,
    nsweeps=6,
    seed=1234,
    verbose=True
)

E_mpo_a = expectation_value_mpo(mps_a, mpo_a).real
E_exact_a = -(2.0 / 3.0) * (L - 1)

print("\nAKLT <H> from MPO contraction =", E_mpo_a)
print("AKLT exact open-chain energy  =", E_exact_a)
print("Absolute error                =", abs(E_mpo_a - E_exact_a))

Hfull_a = dense_aklt(L)
psi_a = mps_to_state(mps_a)
E_dense_a = (np.vdot(psi_a, Hfull_a @ psi_a) / np.vdot(psi_a, psi_a)).real
E0_a = np.linalg.eigvalsh(Hfull_a)[0].real

print("AKLT dense energy from final MPS =", E_dense_a)
print("AKLT exact ED ground energy      =", E0_a)
print("Difference to ED                 =", abs(E_dense_a - E0_a))

Sweep  1: local eig E = -3.333333333333, max discarded weight = 8.997e-24
Sweep  2: local eig E = -3.333333333333, max discarded weight = 2.037e-25
Sweep  3: local eig E = -3.333333333333, max discarded weight = 1.418e-24
Sweep  4: local eig E = -3.333333333333, max discarded weight = 9.297e-26
Sweep  5: local eig E = -3.333333333333, max discarded weight = 2.023e-27
Sweep  6: local eig E = -3.333333333333, max discarded weight = 4.168e-26

AKLT <H> from MPO contraction = -3.3333333333333357
AKLT exact open-chain energy  = -3.333333333333333
Absolute error                = 2.6645352591003757e-15
AKLT dense energy from final MPS = -3.3333333333333344
AKLT exact ED ground energy      = -3.333333333333365
Difference to ED                 = 3.064215547965432e-14


In [15]:
# Cell O
L = 6
mpo_a = aklt_mpo(L)
mps_exact = exact_aklt_mps(L)
mps_exact = normalize_mps_by_state(mps_exact)

E_exact_mps = expectation_value_mpo(mps_exact, mpo_a).real
print("Exact AKLT MPS energy =", E_exact_mps)
print("Expected energy       =", -(2.0 / 3.0) * (L - 1))
print("Error                 =", abs(E_exact_mps + (2.0 / 3.0) * (L - 1)))

Exact AKLT MPS energy = -3.333333333333332
Expected energy       = -3.333333333333333
Error                 = 8.881784197001252e-16
